## Route planning

In [1]:
origin = (120.3181,22.58425)   
dest   = (-118.265,33.74021)  

#| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
#| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
#| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |

#| SHANGHAI  | CNSHG     | (31.36636,121.6147) | (121.6147,31.36636)            |
#| NINGBO    | CNNBG     | (29.92654,121.8525) | (121.8525,29.92654)            |
#| ZHOUSHAN  | CNZOS     | (29.92161,122.2104) | (122.2104,29.92161)            |
#| SHENZHEN  | CNSZX     | (22.5045,113.8535)  | (113.8535,22.5045)             |

#| LOS ANGELES | USLAX     | (33.74021,-118.265) | (-118.265,33.74021)            |
#| SEATTLE     | USSEA     | (47.6212,-122.3643) | (-122.3643,47.6212)            |

#| TOKYO     | JPTYO     | (35.61168,139.8268) | (139.8268,35.61168)            |
#| KOBE      | JPUKB     | (34.6867,135.2671)  | (135.2671,34.6867)             |
#| WAKAYAMA  | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |

In [35]:
# === Lazy Visibility Search (Pacific view, great-circle, full-bbox features) ===
# Author: you + ChatGPT
# Note:
#  - Keep your original geodesy, Pacific-view drawing, land buffering & collision layers.
#  - Replace "outer tangent detour" with a Lazy Visibility Search (LVS) graph.
#  - Nodes = {O, D} ∪ {bbox features}; Edges are only validated lazily by visible().
#  - When a candidate shortest path has an invalid edge, mark it BLOCKED (and optionally inject gateways),
#    then replan until a fully valid path is found.
#
# Usage:
#   1) Set origin, dest (lon,lat) below (or call plan_route(...))
#   2) Run the script; it outputs a Folium HTML with Pacific-centered continuous arcs.

from __future__ import annotations
from pathlib import Path
import math
import heapq
import itertools
from typing import List, Tuple, Dict, Optional, Iterable

import fiona
import folium
from networkx import nodes
import shapely
from shapely.geometry import shape, Polygon, Point, LineString, GeometryCollection
from shapely.ops import unary_union
from shapely.prepared import prep
from shapely.strtree import STRtree
from pyproj import Transformer, Geod

# ---------------- Params (edit as needed) ----------------
LAND_PATH   = Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp")
BUFFER_KM   = 5.0                    # inner "no-go" near coast; points inside will be nudged outward
COLLISION_SAFETY_KM = 0.25           # collision dilation for visibility (rigid safety margin)
PAD_DEG     = 6.0                    # bbox padding in degrees
STEP_KM_GEODESIC = 3.0               # great-circle sampling resolution for visibility
DRAW_STEP_KM = 20.0                  # great-circle sampling for drawing
AVOID_KM    = 15.0                   # outer ring where routes/waypoints are allowed

# LVS knobs
NEIGHBOR_K = 24                      # candidate neighbors per node per expansion (plus D)
LVS_MAX_NODES = 4000                 # guard rail for very large bbox
USE_ROTATION_PENALTY = False         # you asked to keep it OFF initially
ROT_LAMBDA = 0.001                   # weight if enabled

# ---------------- CRS & geodesy ----------------
to_m  = Transformer.from_crs("EPSG:4326","EPSG:3857", always_xy=True).transform
to_ll = Transformer.from_crs("EPSG:3857","EPSG:4326", always_xy=True).transform
def to_metric(g): return shapely.ops.transform(to_m, g)
def to_wgs(g):    return shapely.ops.transform(to_ll, g)
GEOD = Geod(ellps="WGS84")


def _fmt_ll(p):
    return f"({p[0]:.5f}, {p[1]:.5f})"  # (lon, lat)

def _print_ll(msg, p):
    print(f"{msg} {_fmt_ll(p)}", flush=True)

def _print_edge(prefix, a, b):
    print(f"{prefix} {_fmt_ll(a)} -> {_fmt_ll(b)}", flush=True)


def geodesic_sample(a: Tuple[float,float], b: Tuple[float,float], step_km: float=STEP_KM_GEODESIC) -> List[Tuple[float,float]]:
    lon1,lat1=a; lon2,lat2=b
    _,_,dist_m = GEOD.inv(lon1,lat1,lon2,lat2)
    n = max(1,int(dist_m/(step_km*1000)))
    pts = GEOD.npts(lon1,lat1,lon2,lat2,n)
    return [(lon1,lat1)] + pts + [(lon2,lat2)]

def great_circle_midpoint(a,b):
    pts = geodesic_sample(a,b,step_km=500.0)
    return pts[len(pts)//2]

def gc_distance_km(a,b) -> float:
    _,_,d = GEOD.inv(a[0],a[1],b[0],b[1])
    return d/1000.0

def bearing_xy(p,q):
    px,py = to_m(p[0],p[1]); qx,qy = to_m(q[0],q[1])
    return math.degrees(math.atan2(qy-py, qx-px)) % 360.0

def angle_diff(a,b):
    return (a-b+540)%360 - 180

# ---------------- Pacific view helpers ----------------
def normalize_lon_to_pacific_view(lon: float) -> float:
    return lon if lon>=0 else lon+360

def draw_gc_polyline_continuous(m, a, b, step_km=DRAW_STEP_KM, **style):
    pts = geodesic_sample(a, b, step_km=step_km)
    folium_coords = []
    for lon, lat in pts:
        lon_pacific = normalize_lon_to_pacific_view(lon)
        folium_coords.append([lat, lon_pacific])
    folium.PolyLine(folium_coords, **style).add_to(m)

def draw_progress(
    progress: Dict,
    nodes: List[Tuple[float,float]],
    origin: Tuple[float,float],
    dest: Tuple[float,float],
    out_html: str = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_progress.html",
    land_geom=None,     # 建議傳 land_raw_wgs（可為 None）
    ring_geom=None,     # 建議傳 RING_WGS（可為 None）
    feature_nodes: Optional[List[Tuple[float,float]]] = None,  # 若要疊上候選特徵點群組
    show_features: bool = True,
):
    """
    將目前 LVS 進度畫成地圖：
      - 已驗證通過的邊（含過去輪次的 free_edges）→ 實線藍色
      - 本輪候選路徑中已通過的前綴 → 實線藍色
      - 本輪候選路徑中尚未驗證的後綴 → 橘色虛線
      - 若 O/D 曾被推移（nodes[0] != origin 或 nodes[1] != dest），補畫接駁段

    參數：
      progress: 由 lazy_visibility_search(progress=...) 維護的 dict
      nodes:    與 progress["nodes_ref"] 同一個 nodes（索引一致）
      origin,dest: 原始輸入 O/D（lon,lat）
      land_geom, ring_geom: 供背景展示；可傳 None 跳過
      feature_nodes: 要疊圖的候選特徵點（例如 convex_peaks+convex）
    """
    # ---- 讀進度 ----
    cand_idx = progress.get("candidate_path", []) or []
    k = int(progress.get("free_prefix_len", 0) or 0)
    free_edges_hist = list(progress.get("free_edges", []) or [])
    O_idx, D_idx = 0, 1  # 你的設計固定 0/1

    # 安全防呆
    if not nodes or len(nodes) < 2:
        print("[draw_progress] nodes 不足，無法畫圖")
        return
    if not cand_idx:
        # 沒有候選路徑就至少把 O/D 畫出來
        cand_idx = [O_idx, D_idx]
        k = 0

    # 地圖中心：取 O-D 大圓中點（太平洋視角）
    mid_lon, mid_lat = great_circle_midpoint(origin, dest)
    center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
    center_lat = mid_lat

    m = folium.Map(
        location=[center_lat, center_lon_pacific],
        zoom_start=4,
        max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
    )

    # 底圖
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
    ).add_to(m)

    # 背景：陸域、5km 緩衝
    if land_geom is not None:
        folium.GeoJson(
            convert_geom_to_pacific(land_geom),
            name="陸地",
            style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
        ).add_to(m)
    if ring_geom is not None:
        folium.GeoJson(
            convert_geom_to_pacific(ring_geom),
            name=f"航道緩衝區 {BUFFER_KM}km",
            style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
        ).add_to(m)

    # O/D 標記
    folium.Marker(
        [origin[1], normalize_lon_to_pacific_view(origin[0])],
        tooltip=f"起點: ({origin[0]:.4f}, {origin[1]:.4f})",
        icon=folium.Icon(color='green', icon='ship', prefix='fa')
    ).add_to(m)
    folium.Marker(
        [dest[1], normalize_lon_to_pacific_view(dest[0])],
        tooltip=f"終點: ({dest[0]:.4f}, {dest[1]:.4f})",
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)

    # 若 O/D 被推移，補畫接駁段（實線藍）
    O_adj = nodes[O_idx]; D_adj = nodes[D_idx]
    if O_adj != origin:
        draw_gc_polyline_continuous(m, origin, O_adj, step_km=DRAW_STEP_KM,
                                    color='#1f77b4', weight=5, opacity=0.9)
    if D_adj != dest:
        draw_gc_polyline_continuous(m, D_adj, dest, step_km=DRAW_STEP_KM,
                                    color='#1f77b4', weight=5, opacity=0.9)

    # 歷史已驗證 FREE 的邊（跨輪次）：實線藍
    # （有些邊可能不在當前候選路徑上，也照畫，幫助你理解探索過的走向）
    for (u, v) in free_edges_hist:
        if u < len(nodes) and v < len(nodes):
            a, b = nodes[u], nodes[v]
            draw_gc_polyline_continuous(
                m, a, b, step_km=DRAW_STEP_KM,
                color='#1f77b4', weight=4, opacity=0.65
            )

    # 本輪候選路徑：前綴已驗證（實線藍）
    if len(cand_idx) >= 2 and k > 0:
        for u, v in zip(cand_idx[:k], cand_idx[1:k+1]):
            if u < len(nodes) and v < len(nodes):
                a, b = nodes[u], nodes[v]
                draw_gc_polyline_continuous(
                    m, a, b, step_km=DRAW_STEP_KM,
                    color='#1f77b4', weight=6, opacity=0.95
                )

    # 本輪候選路徑：後綴未驗證（橘色虛線）
    if len(cand_idx) >= 2 and k < len(cand_idx) - 1:
        for u, v in zip(cand_idx[k:-1], cand_idx[k+1:]):
            if u < len(nodes) and v < len(nodes):
                a, b = nodes[u], nodes[v]
                draw_gc_polyline_continuous(
                    m, a, b, step_km=DRAW_STEP_KM,
                    color='#ff7f0e', weight=5, opacity=0.9, dash_array="10,6"
                )

    # 候選節點點標（可視化目前 path）
    #  - 前綴點：實心藍圈
    #  - 後綴點：橘圈
    for i, idx in enumerate(cand_idx):
        if idx >= len(nodes): 
            continue
        lon, lat = nodes[idx]
        ll = [lat, normalize_lon_to_pacific_view(lon)]
        if i <= k:
            folium.CircleMarker(ll, radius=4, color='#1f77b4', fill=True, fill_opacity=0.9,
                                tooltip=f"cand[{i}] {_fmt_ll((lon,lat))} (已驗證前綴)").add_to(m)
        else:
            folium.CircleMarker(ll, radius=4, color='#ff7f0e', fill=True, fill_opacity=0.7,
                                tooltip=f"cand[{i}] {_fmt_ll((lon,lat))} (待驗證)").add_to(m)

    # 額外：候選特徵點群組（debug 層）
    if show_features and feature_nodes:
        fg_feat = folium.FeatureGroup(name="候選特徵點（凸峰+凸點）", show=False)
        for (lon, lat) in feature_nodes:
            folium.CircleMarker(
                [lat, normalize_lon_to_pacific_view(lon)], radius=3,
                color="#1f77b4", fill=True, fill_opacity=0.8,
                tooltip=f"Feature ({lon:.3f},{lat:.3f})"
            ).add_to(fg_feat)
        fg_feat.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)
    print(f"[draw_progress] 進度地圖已輸出：{out_html}")


# ---------------- IDL-aware dynamic bboxes ----------------
def dynamic_bboxes_idl(origin, dest, pad_deg: float) -> List[Polygon]:
    o_lon = normalize_lon_to_pacific_view(origin[0])
    d_lon = normalize_lon_to_pacific_view(dest[0])
    min_lon = min(o_lon, d_lon) - pad_deg
    max_lon = max(o_lon, d_lon) + pad_deg
    min_lat = min(origin[1], dest[1]) - pad_deg
    max_lat = max(origin[1], dest[1]) + pad_deg

    bboxes=[]
    if min_lon < 0:
        bboxes.append(Polygon([(min_lon,min_lat),(0,min_lat),(0,max_lat),(min_lon,max_lat)]))
        min_lon=0
    if max_lon > 360:
        bboxes.append(Polygon([(0,min_lat),(max_lon-360,min_lat),(max_lon-360,max_lat),(0,max_lat)]))
        max_lon=360

    lon_min_std = min_lon if min_lon <= 180 else min_lon-360
    lon_max_std = max_lon if max_lon <= 180 else max_lon-360

    if lon_min_std <= lon_max_std:
        bboxes.append(Polygon([(lon_min_std,min_lat),(lon_max_std,min_lat),
                               (lon_max_std,max_lat),(lon_min_std,max_lat)]))
    else:
        bboxes.append(Polygon([(lon_min_std,min_lat),(180,min_lat),(180,max_lat),(lon_min_std,max_lat)]))
        bboxes.append(Polygon([(-180,min_lat),(lon_max_std,min_lat),(lon_max_std,max_lat),(-180,max_lat)]))
    return bboxes

def union_lonlat_bboxes(bboxes: List[Polygon]) -> Polygon:
    # just union them (they are already lon/lat)
    if not bboxes: raise ValueError("No bboxes")
    u = unary_union(bboxes)
    if isinstance(u, GeometryCollection):
        # take envelope if weird
        return u.envelope
    return u

def load_polys_in_bboxes(shp_path: Path, bbox_polys: List[Polygon]) -> List[Polygon]:
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            for box in bbox_polys:
                if g.intersects(box):
                    gi = g.intersection(box)
                    try:
                        parts = list(shapely.get_parts(gi))
                    except Exception:
                        parts = list(gi.geoms) if gi.geom_type=="MultiPolygon" else [gi]
                    for p in parts:
                        if not p.is_empty:
                            polys.append(p)
                    break
    return polys

# ---------------- Land layers & spatial indexes ----------------
def build_land_layers(polys: List[Polygon]):
    # Work in metric
    parts_m = [to_metric(p).buffer(0) for p in polys]
    union_m = unary_union(parts_m)
    collision_m = union_m.buffer(COLLISION_SAFETY_KM * 1000.0)
    ring_m      = union_m.buffer(BUFFER_KM * 1000.0)
    return {
        "UNION_M":          union_m,
        "COLLISION_PREP_M": prep(collision_m),
        "COLLISION_WGS":    to_wgs(collision_m),
        "RING_M":           ring_m,
        "RING_WGS":         to_wgs(ring_m),
        "LAND_RAW_WGS":     to_wgs(union_m),
        "LAND_PARTS_M":     parts_m,
        "COLLISION_M":      collision_m,
    }

def build_land_strtree(parts_m: List[shapely.geometry.base.BaseGeometry]) -> STRtree:
    # STRtree on metric parts to quickly shortlist potential intersections
    return STRtree(parts_m)

# ---------------- Nudge helpers ----------------
def nudge_to_ring_if_inside(pt_ll: Tuple[float,float], UNION_M, inner_buffer_km: float = BUFFER_KM, target_offset_km: float = AVOID_KM):
    px, py = to_m(pt_ll[0], pt_ll[1])
    p_m = Point(px, py)
    inner_ring_m  = UNION_M.buffer(inner_buffer_km * 1000.0)
    target_ring_m = UNION_M.buffer(target_offset_km * 1000.0)
    if inner_ring_m.contains(p_m):
        boundary = target_ring_m.boundary if not target_ring_m.boundary.is_empty else target_ring_m
        q_m = shapely.ops.nearest_points(p_m, boundary)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

# ---- Fast nudge: reuse precomputed rings (NO per-call buffer) ----
def nudge_to_ring_if_inside_fast(
    pt_ll,
    inner_ring_m,          # = UNION_M.buffer(BUFFER_KM*1000)   (precomputed)
    target_boundary_m      # = (UNION_M.buffer(AVOID_KM*1000)).boundary  (precomputed)
):
    """
    若點位於「近岸禁入緩衝」(inner ring) 之內，將其推到「外圈緩衝」(target ring) 的邊界上。
    使用預先計算好的圖層，避免每次都 buffer，速度快很多。
    回傳: (新座標, 是否有移動)
    """
    px, py = to_m(pt_ll[0], pt_ll[1])
    p_m = Point(px, py)

    if inner_ring_m.contains(p_m):
        q_m = shapely.ops.nearest_points(p_m, target_boundary_m)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

# ---- Debug 幫手 ------
def create_lvs_progress():
    """
    用於 Jupyter 中斷後仍能畫圖的進度物件。
    - candidate_path: 當輪 A* 的候選節點索引序列（list[int]）
    - free_prefix_len: 當輪候選路徑中，從起點起連續通過可視檢查（FREE）的邊數量
    - free_edges: 累計所有輪已被標記 FREE 的邊 (u,v)（可重複；畫圖前會過濾處理）
    - nodes_ref: 目前 nodes 的參考（list[(lon,lat)]）
    - iter: 當前 LVS 的迭代次數（純資訊）
    """
    return {
        "candidate_path": None,
        "free_prefix_len": 0,
        "free_edges": [],
        "nodes_ref": None,
        "iter": 0,
    }


# ---------------- Visibility (with STRtree fast prefilter) ----------------
def visible(a, b, COLLISION_PREP_M, land_tree: Optional[STRtree]=None) -> bool:
    # Build geodesic polyline and test against collision buffer (metric space)
    ls_ll = LineString(geodesic_sample(a, b, step_km=STEP_KM_GEODESIC))
    ls_m  = to_metric(ls_ll)

    if land_tree is not None:
        # coarse prefilter: only check parts whose bbox intersects ls
        candidates = land_tree.query(ls_m)
        if len(candidates) == 0:
            return True  # no land around; safe
        # quick reject by buffered line vs. nothing? We'll still ask COLLISION_PREP_M.
    return not COLLISION_PREP_M.intersects(ls_m)

# ---------------- Feature extraction (from your second script, no plotting) ----------------
def _bearing_deg(a, b):
    ax, ay = a; bx, by = b
    return (math.degrees(math.atan2(by - ay, bx - ax)) + 360.0) % 360.0

def _angdiff(a, b):
    d = (a - b + 540.0) % 360.0 - 180.0
    return d

def _resample_linestring_m(ls_m: LineString, step_m: float) -> LineString:
    L = ls_m.length
    if L == 0:
        return ls_m
    n = max(4, int(round(L / step_m)))
    d = L / n
    pts = [ls_m.interpolate(i * d) for i in range(n)]
    if pts[0].distance(pts[-1]) > 1e-6:
        pts.append(pts[0])
    return LineString(pts)

def _local_maxima(seq, radius):
    n = len(seq)
    peaks = []
    for i in range(radius, n - radius):
        v = seq[i]
        if all(v > seq[i - k] for k in range(1, radius + 1)) and \
           all(v >= seq[i + k] for k in range(1, radius + 1)):
            peaks.append(i)
    return peaks

def extract_convex_peaks_from_buffer(
    union_ll,
    avoid_km=AVOID_KM,
    resample_step_m=300.0,
    window_km=8.0,
    peak_radius_pts=3,
    min_turn_deg=18.0,
    dedup_m=1200.0
):
    union_m = to_metric(union_ll)
    buf_m   = union_m.buffer(avoid_km * 1000.0)

    if buf_m.geom_type == "Polygon":
        polys_m = [buf_m]
    else:
        polys_m = [p for p in buf_m.geoms if p.geom_type == "Polygon"]

    out_pts = []
    for poly in polys_m:
        ring_raw = LineString(list(poly.exterior.coords))
        ring     = _resample_linestring_m(ring_raw, resample_step_m)
        coords   = list(ring.coords)
        n        = len(coords)
        if n < 8:
            continue

        ccw = Polygon(coords).exterior.is_ccw
        W = max(1, int(round(window_km * 1000.0 / resample_step_m)))
        dturn = [0.0] * n
        for i in range(W, n - W):
            h1 = _bearing_deg(coords[i - W], coords[i])
            h2 = _bearing_deg(coords[i],     coords[i + W])
            dturn[i] = _angdiff(h2, h1)

        score = [max(0.0, v) if ccw else max(0.0, -v) for v in dturn]
        peaks = _local_maxima(score, peak_radius_pts)
        peaks = [i for i in peaks if score[i] >= min_turn_deg]

        kept = []
        for i in sorted(peaks, key=lambda j: -score[j]):
            pi = Point(coords[i])
            if all(pi.distance(Point(coords[k])) > dedup_m for k in kept):
                kept.append(i)

        for i in kept:
            x, y = coords[i]
            lon, lat = to_ll(x, y)
            out_pts.append((lon, lat))
    return out_pts

def _dedup_points_geom(pts: List[Point], tol_m: float) -> List[Point]:
    out=[]
    for p in pts:
        keep=True
        for q in out:
            if p.distance(q) <= tol_m:
                keep=False; break
        if keep: out.append(p)
    return out

def _cum_lengths(coords):
    L=[0.0]
    for (x1,y1),(x2,y2) in zip(coords, coords[1:]):
        L.append(L[-1] + math.hypot(x2-x1, y2-y1))
    return L

def _index_at_arclen(L, idx, s):
    target_back = L[idx] - s
    i_back = idx
    while i_back > 0 and L[i_back-1] > target_back:
        i_back -= 1
    target_fwd = L[idx] + s
    i_fwd = idx
    n = len(L) - 1
    while i_fwd < n and L[i_fwd+1] < target_fwd:
        i_fwd += 1
    return i_back, i_fwd

def _angle_and_prominence(a,b,c):
    ax, ay = a; bx, by = b; cx, cy = c
    v1x, v1y = ax - bx, ay - by
    v2x, v2y = cx - bx, cy - by
    n1 = math.hypot(v1x, v1y); n2 = math.hypot(v2x, v2y)
    if n1 == 0 or n2 == 0:
        return 180.0, 0.0
    cosang = max(-1.0, min(1.0, (v1x*v2x + v1y*v2y) / (n1*n2)))
    ang = math.degrees(math.acos(cosang))
    vx, vy = cx - ax, cy - ay
    vlen = math.hypot(vx, vy)
    if vlen == 0:
        prom = 0.0
    else:
        abx, aby = bx - ax, by - ay
        cross = abs(abx * vy - aby * vx)
        prom = cross / (vlen * vlen)
    return ang, prom

def score_convex_concave_on_ring(
    ring_ls_m: LineString,
    scales_km=(5, 10, 20, 40),
    angle_convex_max=170.0,
    angle_concave_min=210.0,
    min_prom_convex=0.002,
    min_prom_concave=0.002,
    simplify_before=False, simplify_m=800.0,
    dedup_m=800.0,
):
    ls = ring_ls_m
    if simplify_before:
        ls = ring_ls_m.simplify(simplify_m, preserve_topology=False)
        if not ls.is_ring:
            ls = LineString(list(ls.coords) + [ls.coords[0]])

    coords = list(ls.coords)
    n = len(coords)
    if n < 5:
        return [], [], coords

    ccw = Polygon(coords).exterior.is_ccw
    L = _cum_lengths(coords)
    scales_m = [s * 1000.0 for s in scales_km]
    min_angle = [180.0] * n
    max_prom  = [0.0]   * n

    for i in range(n):
        for s in scales_m:
            i_back, i_fwd = _index_at_arclen(L, i, s)
            if i_back == i or i_fwd == i:
                continue
            a = coords[i_back]; b = coords[i]; c = coords[i_fwd]
            ang, prom = _angle_and_prominence(a, b, c)
            if ang < min_angle[i]:
                min_angle[i] = ang
            if prom > max_prom[i]:
                max_prom[i] = prom

    convex_idx, concave_idx = [], []
    for i in range(1, n - 1):
        a = coords[i - 1]; b = coords[i]; c = coords[i + 1]
        v1x, v1y = a[0] - b[0], a[1] - b[1]
        v2x, v2y = c[0] - b[0], c[1] - b[1]
        cross = v1x * v2y - v1y * v2x
        is_concave = (cross < 0) if ccw else (cross > 0)
        ang  = min_angle[i]
        prom = max_prom[i]

        if not is_concave:
            if ang < angle_convex_max and prom >= min_prom_convex:
                convex_idx.append(i)
        else:
            if ang > angle_concave_min and prom >= min_prom_concave:
                concave_idx.append(i)

    def _dedup_by_spacing(idxs):
        kept = []
        taken = [False] * n
        for i in sorted(idxs, key=lambda j: -max_prom[j]):
            if taken[i]:
                continue
            kept.append(i)
            xi, yi = coords[i]
            for j in range(n):
                if taken[j]:
                    continue
                xj, yj = coords[j]
                if math.hypot(xj - xi, yj - yi) <= dedup_m:
                    taken[j] = True
        return kept

    convex_idx  = _dedup_by_spacing(convex_idx)
    concave_idx = _dedup_by_spacing(concave_idx)
    return convex_idx, concave_idx, coords

def extract_feature_points_bbox(
    shp_path: Path,
    bbox_ll_polygon: Polygon,
    avoid_km=AVOID_KM,
    simplify_m=1000.0,
    ANGLE_CONVEX_MAX=170.0,
    ANGLE_CONCAVE_MIN=210.0,
    MIN_PROM_CONVEX=0.002,
    MIN_PROM_CONCAVE=0.002,
    DEDUP_CONVEX_M=800.0,
    DEDUP_CONCAVE_M=800.0,
    ENABLE_UNIFORM=False,
    TARGET_SPACING_KM=25.0,
    N_MIN=8, N_MAX=64,
    PERIM_MIN_KM=20.0,
    AREA_MIN_KM2=5.0,
) -> Dict[str, List[Tuple[float,float]]]:
    # 1) clip land by bbox
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            if g.is_empty: 
                continue
            if g.intersects(bbox_ll_polygon):
                gi = g.intersection(bbox_ll_polygon)
                if not gi.is_empty:
                    polys.append(gi)
    if not polys:
        return {"convex":[], "concave":[], "uniform":[], "convex_peaks":[]}

    union_ll = unary_union(polys)  # lon/lat
    union_m  = to_metric(union_ll)
    buf_m    = union_m.buffer(avoid_km * 1000.0)

    if buf_m.geom_type == "Polygon":
        polys_m = [buf_m]
    else:
        polys_m = [p for p in buf_m.geoms if p.geom_type == "Polygon"]

    convex_pts_m, concave_pts_m, uniform_pts_m = [], [], []
    for poly in polys_m:
        perim_km = poly.exterior.length / 1000.0
        area_km2 = poly.area / 1e6
        if perim_km < PERIM_MIN_KM or area_km2 < AREA_MIN_KM2:
            continue
        ring_raw = shapely.LineString(poly.exterior.coords)
        ring_s   = ring_raw.simplify(simplify_m, preserve_topology=False)
        if not ring_s.is_ring:
            ring_s = shapely.LineString(list(ring_s.coords)+[ring_s.coords[0]])
        convex_idx, concave_idx, coords_used = score_convex_concave_on_ring(
            ring_s,
            scales_km=(5,10,20,40),
            angle_convex_max=ANGLE_CONVEX_MAX,
            angle_concave_min=ANGLE_CONCAVE_MIN,
            min_prom_convex=MIN_PROM_CONVEX,
            min_prom_concave=MIN_PROM_CONCAVE,
            simplify_before=False,
            dedup_m=max(DEDUP_CONVEX_M, DEDUP_CONCAVE_M),
        )
        for i in convex_idx:
            x,y = coords_used[i]; convex_pts_m.append(Point(x,y))
        for i in concave_idx:
            x,y = coords_used[i]; concave_pts_m.append(Point(x,y))

        if ENABLE_UNIFORM:
            n_uniform = max(N_MIN, min(N_MAX, int(perim_km / TARGET_SPACING_KM)))
            L = ring_raw.length
            for i in range(n_uniform):
                uniform_pts_m.append(ring_raw.interpolate(i * L / n_uniform))

    # convex peaks (direction extrema) using original union_ll
    convex_peaks_ll = extract_convex_peaks_from_buffer(
        union_ll=union_ll,
        avoid_km=avoid_km,
        resample_step_m=300.0,
        window_km=8.0,
        peak_radius_pts=3,
        min_turn_deg=18.0,
        dedup_m=1200.0
    )

    # dedup (metric)
    convex_pts_m  = _dedup_points_geom(convex_pts_m,  DEDUP_CONVEX_M)
    concave_pts_m = _dedup_points_geom(concave_pts_m, DEDUP_CONCAVE_M)
    if ENABLE_UNIFORM:
        uniform_pts_m = _dedup_points_geom(uniform_pts_m, 1500.0)

    def _to_ll_list(pts):
        out=[]
        for p in pts:
            lon,lat = to_ll(p.x,p.y)
            out.append((lon,lat))
        return out

    convex_ll   = _to_ll_list(convex_pts_m)
    concave_ll  = _to_ll_list(concave_pts_m)
    uniform_ll  = _to_ll_list(uniform_pts_m) if ENABLE_UNIFORM else []

    return {
        "convex": convex_ll,
        "concave": concave_ll,
        "uniform": uniform_ll,
        "convex_peaks": convex_peaks_ll
    }

# ---------------- Lazy Visibility Search Graph ----------------
def rotation_cost(seq: List[Tuple[float,float]]) -> float:
    rot=0.0
    for a,b,c in zip(seq[:-2], seq[1:-1], seq[2:]):
        h1 = bearing_xy(a,b); h2 = bearing_xy(b,c)
        rot += abs(angle_diff(h2,h1))
    return rot

def edge_cost(a,b, use_rot=False) -> float:
    # base = geodesic km; optional small rotation penalty is handled path-wise, not per single edge
    return gc_distance_km(a,b)

def heuristic(p, D):
    return gc_distance_km(p, D)

def neighbors_of(u_idx: int, nodes: List[Tuple[float,float]], D_idx: int, k=NEIGHBOR_K) -> List[int]:
    # choose k best forward-movers by distance to D (and also closeness to u to avoid far jumps)
    u = nodes[u_idx]; D = nodes[D_idx]
    # Score: distance to u (prefer closer) + (distance to D from v)  (lower better)
    # We'll preselect by distance-to-u to keep locality, but always include D.
    # Compute distances lazily:
    idxs = range(len(nodes))
    scored = []
    for v_idx in idxs:
        if v_idx == u_idx: 
            continue
        # ensure we don't explode; we filter later by lazy validation
        du = gc_distance_km(u, nodes[v_idx])
        dD = gc_distance_km(nodes[v_idx], D)
        score = du + 0.5*dD
        scored.append((score, v_idx))
    scored.sort(key=lambda t:t[0])
    out = [v for _,v in itertools.islice(scored, 0, k)]
    if D_idx not in out:
        out.append(D_idx)
    return out

def lazy_visibility_search(
    nodes: List[Tuple[float,float]],
    O_idx: int,
    D_idx: int,
    visible_fn,
    COLLISION_PREP_M,
    land_tree: Optional[STRtree],
    inject_gateways_fn,
    max_iters: int = 5000,
    progress: Optional[Dict]=None,   # ←① 加上參數
):
    """
    LazySP：每輪先用樂觀邊跑 A*，再逐邊做可視檢查。
    ★ 會在 Jupyter 直接印出目前檢查到的經緯度點位。
    """
    EDGE_STATE: Dict[Tuple[int,int], str] = {}  # 'FREE'|'BLOCKED'
    adj_cache: Dict[int, List[int]] = {}

    # ② progress 初始化
    if progress is None:
        progress = {"iter": 0, "candidate_path": [], "free_prefix_len": 0,
                    "free_edges": [], "nodes_ref": nodes}
    else:
        progress["nodes_ref"] = nodes

    def get_neighbors(u: int) -> List[int]:
        if u not in adj_cache:
            adj_cache[u] = neighbors_of(u, nodes, D_idx, k=NEIGHBOR_K)
        return adj_cache[u]

    def a_star() -> Optional[List[int]]:
        N_local = len(nodes)
        open_heap = []
        INF = 1e18
        g = [INF] * N_local
        parent = [-1] * N_local

        g[O_idx] = 0.0
        h0 = heuristic(nodes[O_idx], nodes[D_idx])
        heapq.heappush(open_heap, (g[O_idx] + h0, O_idx))
        closed = set()

        while open_heap:
            _, u = heapq.heappop(open_heap)
            if u in closed:
                continue
            if u == D_idx:
                path = [u]
                while parent[u] != -1:
                    u = parent[u]
                    path.append(u)
                path.reverse()
                return path

            closed.add(u)
            for v in get_neighbors(u):
                if v >= len(nodes):
                    continue
                if EDGE_STATE.get((u, v)) == 'BLOCKED':
                    continue

                c = edge_cost(nodes[u], nodes[v], USE_ROTATION_PENALTY)

                if v >= len(g):
                    extend_by = v + 1 - len(g)
                    g.extend([INF] * extend_by)
                    parent.extend([-1] * extend_by)

                alt = g[u] + c
                if alt < g[v]:
                    g[v] = alt
                    parent[v] = u
                    f = alt + heuristic(nodes[v], nodes[D_idx])
                    heapq.heappush(open_heap, (f, v))
        return None

    it = 0
    while it < max_iters:
        it += 1
        path = a_star()
        if not path:
            print("[LVS] 找不到候選路徑（圖被 BLOCKED 邊切斷）", flush=True)
            raise RuntimeError("LVS: path not found.")

        # === 插入點 A：更新本輪進度快照 ===
        progress["iter"] = it
        progress["candidate_path"] = list(path)
        progress["free_prefix_len"] = 0

        # 候選摘要
        cand_nodes = [nodes[idx] for idx in path]
        print(f"[LVS] iter {it:03d} | 候選路徑節點數 = {len(cand_nodes)}", flush=True)
        print(f"         節點總數 = {len(nodes)}", flush=True)
        _print_ll("  候選起點", cand_nodes[0])
        _print_ll("  候選終點", cand_nodes[-1])

        # 逐邊驗證
        all_valid = True
        prefix_ok = 0
        for u, v in zip(path[:-1], path[1:]):
            st = EDGE_STATE.get((u, v))
            a = nodes[u]; b = nodes[v]

            if st == 'FREE':
                _print_edge("  檢查邊(快取OK)", a, b)
                prefix_ok += 1
                progress["free_edges"].append((u, v))
                continue

            if st == 'BLOCKED':
                _print_edge("  檢查邊(快取BLOCKED)", a, b)
                all_valid = False
                break

            _print_edge("  檢查邊", a, b)
            if visible_fn(a, b, COLLISION_PREP_M, land_tree):
                EDGE_STATE[(u, v)] = EDGE_STATE[(v, u)] = 'FREE'
                print("    → OK", flush=True)
                prefix_ok += 1
                progress["free_edges"].append((u, v))
            else:
                EDGE_STATE[(u, v)] = EDGE_STATE[(v, u)] = 'BLOCKED'
                print("    → BLOCKED", flush=True)

                # 被擋：注入 gateway 節點
                new_nodes = inject_gateways_fn(a, b)
                if new_nodes:
                    existing = set((round(lon,5), round(lat,5)) for (lon,lat) in nodes)
                    filtered = []
                    for q in new_nodes:
                        key = (round(q[0],5), round(q[1],5))
                        if key not in existing:
                            filtered.append(q)
                            existing.add(key)
                    if filtered:
                        print(f"    新增節點 {len(filtered)} 個：", flush=True)
                        for q in filtered:
                            _print_ll("      +", q)
                        # ⚠ 正確縮排：extend/clear 在 for 迴圈外
                        nodes.extend(filtered)
                        adj_cache.clear()
                    else:
                        print("    無新節點可加入（皆為重複點）", flush=True)

                all_valid = False
                break  # 停止本輪驗證，回去重規劃

        # 回寫本輪已通過的前綴長度（給中斷時畫圖用）
        progress["free_prefix_len"] = prefix_ok

        if all_valid:
            print("[LVS] 成功：路徑全通過驗證，最終節點序列：", flush=True)
            for i, idx in enumerate(path):
                _print_ll(f"  [{i:02d}]", nodes[idx])
            return path

    raise RuntimeError("LVS: 超過最大迭代次數仍未找到有效路徑")





# ---------------- Gateway injection (simple + robust) ----------------
def make_inject_gateways_fn(
    UNION_M,
    features_index_ll,
    take_each=3,
    inner_ring_m=None,
    target_boundary_m=None
):
    pool = list(features_index_ll.get("convex_peaks", [])) + list(features_index_ll.get("convex", []))

    def f(u_ll, v_ll):
        if not pool:
            return []

        seg_ll = LineString(geodesic_sample(u_ll, v_ll, step_km=STEP_KM_GEODESIC))
        seg_m  = to_metric(seg_ll)

        # 以距離這條被擋邊的最近度排序
        cand = []
        for (lon, lat) in pool:
            px, py = to_m(lon, lat)
            p = Point(px, py)
            cand.append((seg_m.distance(p), (lon, lat)))
        cand.sort(key=lambda t: t[0])

        # 先取前 take_each 個候選，推離岸
        new = []
        for _, pt in itertools.islice(cand, 0, take_each):
            q, _moved = nudge_to_ring_if_inside_fast(pt, inner_ring_m, target_boundary_m)
            new.append(q)

        # 1) 本批去重
        out = []
        seen_local = set()
        for lon, lat in new:
            key = (round(lon, 5), round(lat, 5))
            if key in seen_local:
                continue
            seen_local.add(key)
            out.append((lon, lat))

        # 2) 與既有 nodes 去重（關鍵）
        # 這裡從閉包外讀不到 nodes，所以改成在 lazy_visibility_search() 裡過濾（見下方補丁）
        return out

    return f



# ---------------- Utility: convert land geom to Pacific view for folium ----------------
def convert_geom_to_pacific(geom):
    from shapely.geometry import mapping
    geom_dict = mapping(geom)
    def convert_coords(coords):
        if isinstance(coords[0], (list, tuple)):
            return [convert_coords(c) for c in coords]
        else:
            lon, lat = coords[0], coords[1]
            lon_pacific = normalize_lon_to_pacific_view(lon)
            return [lon_pacific, lat]
    if geom_dict['type'] == 'Polygon':
        geom_dict['coordinates'] = [convert_coords(ring) for ring in geom_dict['coordinates']]
    elif geom_dict['type'] == 'MultiPolygon':
        geom_dict['coordinates'] = [[convert_coords(ring) for ring in poly] for poly in geom_dict['coordinates']]
    return geom_dict

# ---------------- Orchestrator ----------------
def plan_route(
    origin: Tuple[float,float],
    dest: Tuple[float,float],
    land_path: Path = LAND_PATH,
    out_html: str = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_pacific_LVS.html",
    add_feature_layer: bool = True,
):
    print(f"起點: {origin}")
    print(f"終點: {dest}")
    print(f"太平洋視角: 起點={normalize_lon_to_pacific_view(origin[0]):.2f}°, "
          f"終點={normalize_lon_to_pacific_view(dest[0]):.2f}°")

    # 1) Load land inside IDL-aware bboxes
    bboxes = dynamic_bboxes_idl(origin, dest, pad_deg=PAD_DEG)
    polys  = load_polys_in_bboxes(land_path, bboxes)
    assert len(polys)>0, "No land polygons found in bbox"
    layers = build_land_layers(polys)
    UNION_M          = layers["UNION_M"]
    COLLISION_PREP_M = layers["COLLISION_PREP_M"]
    COLLISION_WGS    = layers["COLLISION_WGS"]
    RING_WGS         = layers["RING_WGS"]
    land_raw_wgs     = layers["LAND_RAW_WGS"]
    LAND_PARTS_M     = layers["LAND_PARTS_M"]

    INNER_RING_M = layers["RING_M"]  # 5km 內圈（已在 build_land_layers 算過）
    TARGET_RING_M = UNION_M.buffer(AVOID_KM * 1000.0)   # 15km 外圈（只算一次！）
    TARGET_BOUNDARY_M = TARGET_RING_M.boundary

    land_tree = build_land_strtree(LAND_PARTS_M)

    # 2) Nudge O/D
    origin_adj, moved_o = nudge_to_ring_if_inside_fast(origin, INNER_RING_M, TARGET_BOUNDARY_M)
    dest_adj,   moved_d = nudge_to_ring_if_inside_fast(dest,   INNER_RING_M, TARGET_BOUNDARY_M)

    if moved_o:
        _print_edge("[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈", origin, origin_adj)
    if moved_d:
        _print_edge("[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈", dest, dest_adj)

    # 3) Extract features for the whole bbox union
    bbox_union_ll = union_lonlat_bboxes(bboxes)
    feat = extract_feature_points_bbox(
        shp_path=land_path,
        bbox_ll_polygon=bbox_union_ll,
        avoid_km=AVOID_KM,
        simplify_m=1000.0,
        ANGLE_CONVEX_MAX=170.0,
        ANGLE_CONCAVE_MIN=210.0,
        MIN_PROM_CONVEX=0.002,
        MIN_PROM_CONCAVE=0.002,
        ENABLE_UNIFORM=False,  # per your choice
        PERIM_MIN_KM=20.0,
        AREA_MIN_KM2=5.0
    )
    # feature combo per your decision: convex + convex_peaks
    feature_nodes = list(feat["convex_peaks"]) + list(feat["convex"])

    # 4) Build initial node list and nudge them all
    base_nodes = [origin_adj, dest_adj] + feature_nodes
    nodes=[]
    for p in base_nodes:
        q,_ = nudge_to_ring_if_inside_fast(p, INNER_RING_M, TARGET_BOUNDARY_M)
        nodes.append(q)
    if len(nodes) > LVS_MAX_NODES:
        print(f"[WARN] nodes truncated from {len(nodes)} to {LVS_MAX_NODES}")
        nodes = nodes[:LVS_MAX_NODES]
    O_idx=0; D_idx=1

    # 5) Make injection function for blocked edges
    inject_fn = make_inject_gateways_fn(
        UNION_M,
        {"convex_peaks": feat["convex_peaks"], "convex": feat["convex"]},
        take_each=3,
        inner_ring_m=INNER_RING_M,
        target_boundary_m=TARGET_BOUNDARY_M
    )

    # 6) Run Lazy Visibility Search
    def visible_wrapper(a,b, COLLISION_PREP_M, land_tree):
        return visible(a,b, COLLISION_PREP_M, land_tree)

    print("\n開始 Lazy Visibility Search (LVS)...")
    progress = {"iter":0, "candidate_path":[], "free_prefix_len":0, "free_edges":[], "nodes_ref": None}
    path_idx = lazy_visibility_search(
        nodes, O_idx, D_idx, visible_wrapper, COLLISION_PREP_M, land_tree, inject_fn, max_iters=5000, progress=progress
    )
    print(f"[OK] LVS path with {len(path_idx)} nodes")

    # 7) Build Folium (Pacific view)
    mid_lon, mid_lat = great_circle_midpoint(origin, dest)
    center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
    center_lat = mid_lat

    m = folium.Map(
        location=[center_lat, center_lon_pacific],
        zoom_start=3,
        max_bounds=False, world_copy_jump=False, no_wrap=False, min_lon=0, max_lon=360
    )
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Ocean Basemap', overlay=False, control=True, no_wrap=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='© OpenStreetMap', name='OpenStreetMap', overlay=False, control=True, no_wrap=False
    ).add_to(m)

    # Land + 5km ring
    folium.GeoJson(
        convert_geom_to_pacific(land_raw_wgs),
        name="陸地",
        style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
    ).add_to(m)
    folium.GeoJson(
        convert_geom_to_pacific(RING_WGS),
        name=f"航道緩衝區 {BUFFER_KM}km",
        style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
    ).add_to(m)

    # Markers: O/D
    origin_pacific = [origin[1], normalize_lon_to_pacific_view(origin[0])]
    dest_pacific   = [dest[1],   normalize_lon_to_pacific_view(dest[0])]
    folium.Marker(
        origin_pacific,
        tooltip=f"起點: <br>({origin[0]:.2f}°, {origin[1]:.2f}°)",
        icon=folium.Icon(color='green', icon='ship', prefix='fa')
    ).add_to(m)
    folium.Marker(
        dest_pacific,
        tooltip=f"終點: <br>({dest[0]:.2f}°, {dest[1]:.2f}°)",
        icon=folium.Icon(color='red', icon='anchor', prefix='fa')
    ).add_to(m)

    # reference great-circle
    draw_gc_polyline_continuous(
        m, origin, dest, step_km=80.0,
        color='gray', weight=2, opacity=0.4, dash_array="8,4"
    )

    # Candidate feature nodes layer (for debugging)
    if add_feature_layer:
        fg_feat = folium.FeatureGroup(name="候選特徵點（凸峰+凸點）", show=False)
        for (lon,lat) in feature_nodes:
            folium.CircleMarker(
                [lat, normalize_lon_to_pacific_view(lon)], radius=3,
                color="#1f77b4", fill=True, fill_opacity=0.8,
                tooltip=f"Feature ({lon:.3f},{lat:.3f})"
            ).add_to(fg_feat)
        fg_feat.add_to(m)

    # 8) Draw final path as continuous blue GC segments
    final_segments=[]
    # (A) 若起點被推移，先把 origin→origin_adj 這段納入與繪圖
    if moved_o and origin != origin_adj:
        final_segments.append((origin, origin_adj))
        draw_gc_polyline_continuous(
            m, origin, origin_adj, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )

    # (B) 主路徑（origin_adj … dest_adj）
    for u, v in zip(path_idx[:-1], path_idx[1:]):
        a = nodes[u]; b = nodes[v]
        final_segments.append((a, b))
        draw_gc_polyline_continuous(
            m, a, b, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )

    # (C) 若終點被推移，最後把 dest_adj→dest 也納入與繪圖
    if moved_d and dest_adj != dest:
        final_segments.append((dest_adj, dest))
        draw_gc_polyline_continuous(
            m, dest_adj, dest, step_km=DRAW_STEP_KM,
            color='#1f77b4', weight=5, opacity=0.9
        )
    # 9) Waypoint markers (exclude ends)
    for i, idx in enumerate(path_idx[1:-1], 1):
        wp = nodes[idx]
        wp_pacific = [wp[1], normalize_lon_to_pacific_view(wp[0])]
        folium.CircleMarker(
            wp_pacific, radius=4,
            tooltip=f"航點 {i}<br>({wp[0]:.2f}°, {wp[1]:.2f}°)",
            color='blue', fill=True, fill_opacity=0.8
        ).add_to(m)

    # 存檔 & 總結
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(out_html)

    total_km = sum(gc_distance_km(a, b) for (a, b) in final_segments)

    print(f"\n已儲存至: {out_html}")
    print(f"總航程: {total_km:.1f} 公里")
    print(f"節點數: {len(nodes)} / 路徑節點: {len(path_idx)}")

# 額外資訊：推移距離（若有）
    if moved_o:
        print(f"[INFO] 起點推移距離 ≈ {gc_distance_km(origin, origin_adj):.2f} km")
    if moved_d:
        print(f"[INFO] 終點推移距離 ≈ {gc_distance_km(dest, dest_adj):.2f} km")



# ---------------- Example call (uncomment to run) ----------------
if __name__ == "__main__":
    # Example ports (Tokyo -> Los Angeles), adjust as you like:
    # origin = (139.8268, 35.61168)
    # dest   = (-118.265, 33.74021)

    # Taiwan → Japan sample (you can replace):
    origin = (120.3181,22.58425)   
    dest   = (103.5, 10.633333)   
    #| KAOHSIUNG  | TWKHH     | (22.58425,120.3181)  | (120.3181,22.58425)            |
    #| TAICHUNG   | TWTXG     | (24.255,120.5075)    | (120.5075,24.255)              |
    #| KEELUNG    | TWKEL     | (25.14515,121.7532)  | (121.7532,25.14515)            |
    #| SHANGHAI  | CNSHG     | (31.36636,121.6147) | (121.6147,31.36636)            |
    #| NINGBO    | CNNBG     | (29.92654,121.8525) | (121.8525,29.92654)            |
    #| ZHOUSHAN  | CNZOS     | (29.92161,122.2104) | (122.2104,29.92161)            |
    #| SHENZHEN  | CNSZX     | (22.5045,113.8535)  | (113.8535,22.5045)             |
    #| LOS ANGELES | USLAX     | (33.74021,-118.265) | (-118.265,33.74021)            |
    #| SEATTLE     | USSEA     | (47.6212,-122.3643) | (-122.3643,47.6212)            |
    #| TOKYO     | JPTYO     | (35.61168,139.8268) | (139.8268,35.61168)            |
    #| KOBE      | JPUKB     | (34.6867,135.2671)  | (135.2671,34.6867)             |
    #| WAKAYAMA  | JPUKW     | (33.468056,135.828056) | (135.828056,33.468056)       |
    #| BUSAN     | KRPUS     | (35.116667,129.033333) | (129.033333,35.116667)       |
    #|  Sihanoukville| KHKOS  |                      |   (103.5, 10.633333)       |

    plan_route(origin, dest)


起點: (120.3181, 22.58425)
終點: (103.5, 10.633333)
太平洋視角: 起點=120.32°, 終點=103.50°
[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈 (120.31810, 22.58425) -> (120.18383, 22.49683)
[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈 (103.50000, 10.63333) -> (103.44603, 10.48629)

開始 Lazy Visibility Search (LVS)...
[LVS] iter 001 | 候選路徑節點數 = 2
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 002 | 候選路徑節點數 = 3
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (120.21215, 22.35222)
    → OK
  檢查邊 (120.21215, 22.35222) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 003 | 候選路徑節點數 = 3
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (120.21064, 22.33983)
    → OK
  檢查邊 (120.21064, 22.33983) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 004 | 候選路

RuntimeError: LVS: path not found.

In [36]:
try:
    plan_route(origin, dest)  # 你本來的呼叫
except KeyboardInterrupt:
    # 這裡用你在 plan_route 中可取得的物件來畫（建議把需要的東西暴露出來或存在全域/外層）
    draw_progress(
        progress=progress,         # 傳入 lazy_visibility_search 使用的 progress 物件
        nodes=nodes,               # 同一份 nodes
        origin=origin, 
        dest=dest,
        out_html=r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_progress.html",
        land_geom=land_raw_wgs,    # 若不方便取到，傳 None 就好
        ring_geom=RING_WGS,
        feature_nodes=feature_nodes,   # 若要一起看
        show_features=True
    )


起點: (120.3181, 22.58425)
終點: (103.5, 10.633333)
太平洋視角: 起點=120.32°, 終點=103.50°
[INFO] 起點在 5km 緩衝內 → 已推到 15km 外圈 (120.31810, 22.58425) -> (120.18383, 22.49683)
[INFO] 終點在 5km 緩衝內 → 已推到 15km 外圈 (103.50000, 10.63333) -> (103.44603, 10.48629)

開始 Lazy Visibility Search (LVS)...
[LVS] iter 001 | 候選路徑節點數 = 2
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 002 | 候選路徑節點數 = 3
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (120.21215, 22.35222)
    → OK
  檢查邊 (120.21215, 22.35222) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 003 | 候選路徑節點數 = 3
         節點總數 = 3883
  候選起點 (120.18383, 22.49683)
  候選終點 (103.44603, 10.48629)
  檢查邊 (120.18383, 22.49683) -> (120.21064, 22.33983)
    → OK
  檢查邊 (120.21064, 22.33983) -> (103.44603, 10.48629)
    → BLOCKED
    無新節點可加入（皆為重複點）
[LVS] iter 004 | 候選路

RuntimeError: LVS: path not found.

In [37]:
# === 跨國際換日線的連續航線繪製（太平洋中心投影）===
# 關鍵：使用 center_lon = 180，讓航線在視覺上完全連續

from pathlib import Path
import fiona
from shapely.geometry import shape, Polygon, Point, LineString
import shapely
from shapely.ops import unary_union
from shapely.prepared import prep
from shapely.ops import nearest_points
from pyproj import Transformer, Geod
import folium
import math

# ---------------- Params ----------------
LAND_PATH   = Path(r"C:\Users\slab\Desktop\Slab Project\Stage1\data\Land\ne_10m_land.shp")
BUFFER_KM   = 15.0
COLLISION_SAFETY_KM = 0.25
PAD_DEG     = 6.0
STEP_KM_GEODESIC = 3.0
DRAW_STEP_KM = 20.0
MAX_STEPS   = None

# ---------- CRS & geodesy ----------
to_m  = Transformer.from_crs("EPSG:4326","EPSG:3857", always_xy=True).transform
to_ll = Transformer.from_crs("EPSG:3857","EPSG:4326",  always_xy=True).transform
def to_metric(g): return shapely.ops.transform(to_m, g)
def to_wgs(g):    return shapely.ops.transform(to_ll, g)
GEOD = Geod(ellps="WGS84")

# ---------- Great-circle helpers ----------
def geodesic_sample(a,b,step_km=STEP_KM_GEODESIC):
    lon1,lat1=a; lon2,lat2=b
    _,_,dist_m = GEOD.inv(lon1,lat1,lon2,lat2)
    n = max(1,int(dist_m/(step_km*1000)))
    pts = GEOD.npts(lon1,lat1,lon2,lat2,n)
    return [(lon1,lat1)] + pts + [(lon2,lat2)]

def great_circle_midpoint(a,b):
    pts = geodesic_sample(a,b,step_km=500.0)
    return pts[len(pts)//2]

def normalize_lon_to_pacific_view(lon):
    """
    將經度轉換到太平洋中心視角 [0, 360)
    - 日本 141° → 141°
    - 加州 -132° → 228°
    這樣它們之間就是連續的了
    """
    return lon if lon >= 0 else lon + 360

def draw_gc_polyline_continuous(m, a, b, step_km=DRAW_STEP_KM, **style):
    """
    繪製連續的大圓航線（太平洋視角，無斷裂）
    """
    pts = geodesic_sample(a, b, step_km=step_km)
    
    # 轉換為太平洋視角 [0, 360)
    folium_coords = []
    for lon, lat in pts:
        lon_pacific = normalize_lon_to_pacific_view(lon)
        # Folium 需要 [lat, lon] 格式
        folium_coords.append([lat, lon_pacific])
    
    # 一次性繪製整條線（不斷開）
    folium.PolyLine(folium_coords, **style).add_to(m)

def bearing_xy(p,q):
    px,py = to_m(p[0],p[1]); qx,qy = to_m(q[0],q[1])
    return math.degrees(math.atan2(qy-py, qx-px)) % 360.0

def angle_diff(a,b):
    return (a-b+540)%360 - 180

# ---------- IDL-aware dynamic bboxes ----------
def dynamic_bboxes_idl(origin, dest, pad_deg):
    """
    為太平洋視角創建 BBOX（使用 [0, 360) 經度範圍）
    """
    o_lon = normalize_lon_to_pacific_view(origin[0])
    d_lon = normalize_lon_to_pacific_view(dest[0])
    
    min_lon = min(o_lon, d_lon) - pad_deg
    max_lon = max(o_lon, d_lon) + pad_deg
    min_lat = min(origin[1], dest[1]) - pad_deg
    max_lat = max(origin[1], dest[1]) + pad_deg
    
    bboxes = []
    
    # 如果 BBOX 跨越 360°/0° 邊界
    if min_lon < 0:
        # 左側部分（轉回負數給 shapefile）
        bboxes.append(Polygon([
            (min_lon, min_lat), (0, min_lat), 
            (0, max_lat), (min_lon, max_lat)
        ]))
        min_lon = 0
    
    if max_lon > 360:
        # 右側部分
        bboxes.append(Polygon([
            (0, min_lat), (max_lon - 360, min_lat),
            (max_lon - 360, max_lat), (0, max_lat)
        ]))
        max_lon = 360
    
    # 主要部分（轉回 [-180, 180] 給 shapefile）
    lon_min_std = min_lon if min_lon <= 180 else min_lon - 360
    lon_max_std = max_lon if max_lon <= 180 else max_lon - 360
    
    if lon_min_std <= lon_max_std:
        bboxes.append(Polygon([
            (lon_min_std, min_lat), (lon_max_std, min_lat),
            (lon_max_std, max_lat), (lon_min_std, max_lat)
        ]))
    else:
        # 跨越 180° 的情況
        bboxes.append(Polygon([
            (lon_min_std, min_lat), (180, min_lat),
            (180, max_lat), (lon_min_std, max_lat)
        ]))
        bboxes.append(Polygon([
            (-180, min_lat), (lon_max_std, min_lat),
            (lon_max_std, max_lat), (-180, max_lat)
        ]))
    
    return bboxes

def load_polys_in_bboxes(shp_path: Path, bbox_polys):
    polys=[]
    with fiona.open(shp_path) as src:
        for feat in src:
            g = shape(feat["geometry"])
            for box in bbox_polys:
                if g.intersects(box):
                    gi = g.intersection(box)
                    try:
                        parts = list(shapely.get_parts(gi))
                    except Exception:
                        parts = list(gi.geoms) if gi.geom_type=="MultiPolygon" else [gi]
                    for p in parts:
                        if not p.is_empty:
                            polys.append(p)
                    break
    return polys

# ---------- Land layers ----------
def build_land_layers(polys):
    parts_m = [to_metric(p).buffer(0) for p in polys]
    union_m = unary_union(parts_m)
    collision_m = union_m.buffer(COLLISION_SAFETY_KM * 1000.0)
    ring_m      = union_m.buffer(BUFFER_KM * 1000.0)
    return {
        "UNION_M":          union_m,
        "COLLISION_PREP_M": prep(collision_m),
        "COLLISION_WGS":    to_wgs(collision_m),
        "RING_M":           ring_m,
        "RING_WGS":         to_wgs(ring_m),
        "LAND_RAW_WGS":     to_wgs(union_m),
    }

# ---------- First blocking polygon ----------
def first_blocking_polygon(O, D, polys, COLLISION_PREP_M):
    line_ll = LineString(geodesic_sample(O, D))
    line_m  = to_metric(line_ll)
    inter = line_m.intersection(COLLISION_PREP_M.context)
    if inter.is_empty:
        return None

    def _candidate_points(g):
        if g.geom_type == "Point":
            return [g]
        if g.geom_type in ("MultiPoint", "GeometryCollection"):
            return [x for x in g.geoms if x.geom_type == "Point"]
        if g.geom_type in ("LineString", "LinearRing"):
            cs = list(g.coords); return [Point(cs[0]), Point(cs[-1])]
        if g.geom_type == "MultiLineString":
            pts=[]
            for ls in g.geoms:
                cs=list(ls.coords); pts += [Point(cs[0]), Point(cs[-1])]
            return pts
        try:
            return [g.representative_point()]
        except Exception:
            return []

    cand_pts = _candidate_points(inter) or [inter.representative_point()]
    hit_pt_m = min(cand_pts, key=lambda p: line_m.project(p))

    best_P, best_d = None, None
    for P in polys:
        Pm = to_metric(P).buffer(0)
        P_buf_m = Pm.buffer(COLLISION_SAFETY_KM * 1000.0)
        if P_buf_m.contains(hit_pt_m) or P_buf_m.intersects(hit_pt_m):
            return P
        d = P_buf_m.distance(hit_pt_m)
        if (best_d is None) or (d < best_d):
            best_d, best_P = d, P
    return best_P

# ---------- Offset ring + routing ----------
def polygon_offset_ring_lonlat(P, offset_km=BUFFER_KM):
    Pm   = to_metric(P)
    ring = Pm.buffer(offset_km*1000.0).exterior
    ring_ll = to_wgs(ring)
    coords = list(ring_ll.coords)[:-1]
    return coords

def tangent_pair(X, cut_pts, ref_point, visible_fn):
    ang_c = bearing_xy(X, ref_point)
    vis=[]
    for i,P in enumerate(cut_pts):
        if visible_fn(X,P):
            ang_p = bearing_xy(X,P)
            vis.append((i,P, angle_diff(ang_p, ang_c)))
    if not vis: return None, None
    left  = max(vis, key=lambda t:t[2])
    right = min(vis, key=lambda t:t[2])
    return left[0], right[0]

def arc_indices(i,j,n,dirsign):
    idx=[]; k=i
    while True:
        idx.append(k)
        if k==j: break
        k=(k+dirsign)%n
        if len(idx)>n+2: break
    return idx

def get_arc_points(idxs, ring_coords, step_km=STEP_KM_GEODESIC):
    pts = [ring_coords[i] for i in idxs]
    out=[pts[0]]
    for a,b in zip(pts[:-1], pts[1:]):
        out += geodesic_sample(a,b, step_km=step_km)[1:]
    return out

def total_length_km(seq):
    s=0.0
    for a,b in zip(seq[:-1], seq[1:]):
        _,_,d = GEOD.inv(a[0],a[1],b[0],b[1]); s+=d/1000.0
    return s

def best_step_for_polygon(O, D, P, visible_fn):
    ring_coords = polygon_offset_ring_lonlat(P, BUFFER_KM)
    n = len(ring_coords)
    ref = Polygon(ring_coords).centroid
    ref_point = (ref.x, ref.y)

    li, ri = tangent_pair(O, ring_coords, ref_point, visible_fn)
    lj, rj = tangent_pair(D, ring_coords, ref_point, visible_fn)
    if None in (li,ri,lj,rj):
        vis_idx = [i for i,p in enumerate(ring_coords) if visible_fn(O,p)]
        if not vis_idx:
            return {"status":"fail","next_wp":None,"path":[O]}
        best_i = min(vis_idx, key=lambda i: GEOD.inv(ring_coords[i][0], ring_coords[i][1], D[0],D[1])[2])
        return {"status":"fallback","next_wp":ring_coords[best_i],"path":[O, ring_coords[best_i]]}

    combos = [(li, rj, +1), (ri, lj, -1)]
    cand=[]
    for i_start, j_end, dirsign in combos:
        idxs = arc_indices(i_start, j_end, n, dirsign)
        arc_pts = get_arc_points(idxs, ring_coords)
        if not (visible_fn(O, arc_pts[0]) and visible_fn(arc_pts[-1], D)):
            continue
        seq = [O] + arc_pts + [D]
        length = total_length_km(seq)
        rot=0.0
        for a,b,c in zip(seq[:-2], seq[1:-1], seq[2:]):
            h1 = bearing_xy(a,b); h2=bearing_xy(b,c)
            rot += abs(angle_diff(h2,h1))
        score = length + 0.001*rot
        cand.append((score, seq))
    if not cand:
        vis_idx = [i for i,p in enumerate(ring_coords) if visible_fn(O,p)]
        if not vis_idx:
            return {"status":"fail","next_wp":None,"path":[O]}
        best_i = min(vis_idx, key=lambda i: GEOD.inv(ring_coords[i][0], ring_coords[i][1], D[0],D[1])[2])
        return {"status":"fallback","next_wp":ring_coords[best_i],"path":[O, ring_coords[best_i]]}

    cand.sort(key=lambda x:x[0])
    best_seq = cand[0][1]
    next_wp  = best_seq[1]
    return {"status":"arc","next_wp":next_wp,"path":best_seq}

def nudge_to_ring_if_inside(pt_ll, UNION_M, ring_offset_km=15.05):
    px, py = to_m(pt_ll[0], pt_ll[1]); p_m = Point(px, py)
    ring_m = UNION_M.buffer(BUFFER_KM * 1000.0)
    if ring_m.contains(p_m):
        ring_eps_m = UNION_M.buffer(ring_offset_km * 1000.0)
        q_m = nearest_points(p_m, ring_eps_m.exterior)[1]
        q_llx, q_lly = to_ll(q_m.x, q_m.y)
        return (q_llx, q_lly), True
    return pt_ll, False

# ===================== DEMO =====================
origin = (120.3181,22.58425)   
dest   = (103.5, 10.633333) 

print(f"起點: {origin} ")
print(f"終點: {dest} ")
print(f"太平洋視角: 起點={normalize_lon_to_pacific_view(origin[0]):.2f}°, "
      f"終點={normalize_lon_to_pacific_view(dest[0]):.2f}°")

# 1) 載入陸地（使用太平洋視角的 BBOX）
bboxes = dynamic_bboxes_idl(origin, dest, pad_deg=PAD_DEG)
polys  = load_polys_in_bboxes(LAND_PATH, bboxes)
print(f"載入 {len(polys)} 個陸地多邊形")
assert len(polys)>0, "No land polygons found"

# 2) 建立圖層
layers = build_land_layers(polys)
UNION_M          = layers["UNION_M"]
COLLISION_PREP_M = layers["COLLISION_PREP_M"]
COLLISION_WGS    = layers["COLLISION_WGS"]
RING_M           = layers["RING_M"]
RING_WGS         = layers["RING_WGS"]
land_raw_wgs     = layers["LAND_RAW_WGS"]

# 3) 起點調整
origin_adj, moved_o = nudge_to_ring_if_inside(origin, UNION_M)
dest_adj, moved_d = dest, False

# 4) 可視判定
def visible(a, b):
    line_ll = LineString(geodesic_sample(a, b, step_km=STEP_KM_GEODESIC))
    line_m  = to_metric(line_ll)
    return not COLLISION_PREP_M.intersects(line_m)

# 5) 計算地圖中心（使用航線中點）
mid_lon, mid_lat = great_circle_midpoint(origin, dest)
center_lon_pacific = normalize_lon_to_pacific_view(mid_lon)
center_lat = mid_lat

print(f"地圖中心: ({center_lon_pacific:.2f}°, {center_lat:.2f}°)")

# 6) 創建地圖（太平洋中心視角 + 循環顯示）
m = folium.Map(
    location=[center_lat, center_lon_pacific], 
    zoom_start=3,
    max_bounds=False,  # 不限制視圖範圍
    world_copy_jump=False,  # 不跳躍到地圖副本
    no_wrap=False,  # 允許經度循環
    min_lon=0,      # 設定經度範圍下限
    max_lon=360     # 設定經度範圍上限（允許超過 180°）
)

# 使用支援太平洋視角的底圖（不會在邊緣斷裂）
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Ocean_Basemap/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Ocean Basemap',
    overlay=False,
    control=True,
    no_wrap=False  # 底圖也允許循環
).add_to(m)

# 備用底圖
folium.TileLayer(
    tiles='https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png',
    attr='© OpenStreetMap',
    name='OpenStreetMap',
    overlay=False,
    control=True,
    no_wrap=False  # 底圖也允許循環
).add_to(m)

# 畫陸地圖層（需要轉換座標）
def convert_geom_to_pacific(geom):
    """將幾何體的經度轉換到太平洋視角"""
    from shapely.geometry import mapping, shape as shapely_shape
    import copy
    
    geom_dict = mapping(geom)
    
    def convert_coords(coords):
        if isinstance(coords[0], (list, tuple)):
            return [convert_coords(c) for c in coords]
        else:
            lon, lat = coords[0], coords[1]
            lon_pacific = normalize_lon_to_pacific_view(lon)
            return [lon_pacific, lat]
    
    if geom_dict['type'] == 'Polygon':
        geom_dict['coordinates'] = [convert_coords(ring) for ring in geom_dict['coordinates']]
    elif geom_dict['type'] == 'MultiPolygon':
        geom_dict['coordinates'] = [[convert_coords(ring) for ring in poly] for poly in geom_dict['coordinates']]
    
    return geom_dict

# 畫陸地
folium.GeoJson(
    convert_geom_to_pacific(land_raw_wgs),
    name="陸地",
    style_function=lambda x: {"color":"#2ca02c","weight":1,"fillOpacity":0.15}
).add_to(m)

folium.GeoJson(
    convert_geom_to_pacific(RING_WGS),
    name=f"航道緩衝區 {BUFFER_KM}km",
    style_function=lambda x: {"color":"#6a5acd","weight":2,"fillOpacity":0.05}
).add_to(m)

# 起訖點（轉換為太平洋視角）
origin_pacific = [origin[1], normalize_lon_to_pacific_view(origin[0])]
dest_pacific = [dest[1], normalize_lon_to_pacific_view(dest[0])]

folium.Marker(
    origin_pacific, 
    tooltip=f"起點: 日本<br>({origin[0]:.2f}°, {origin[1]:.2f}°)",
    icon=folium.Icon(color='green', icon='ship', prefix='fa')
).add_to(m)

folium.Marker(
    dest_pacific, 
    tooltip=f"終點: 加州<br>({dest[0]:.2f}°, {dest[1]:.2f}°)",
    icon=folium.Icon(color='red', icon='anchor', prefix='fa')
).add_to(m)

# 參考大圓（虛線）
draw_gc_polyline_continuous(
    m, origin, dest, step_km=80.0,
    color='gray', weight=2, opacity=0.4, dash_array="8,4"
)

# 若有調整起點
if moved_o and origin_adj != origin:
    draw_gc_polyline_continuous(
        m, origin, origin_adj, step_km=5.0,
        color='orange', weight=3, opacity=0.7
    )
    origin_adj_pacific = [origin_adj[1], normalize_lon_to_pacific_view(origin_adj[0])]
    folium.CircleMarker(
        origin_adj_pacific, 
        radius=5,
        tooltip="調整後起點",
        fill=True,
        color='orange',
        fill_opacity=0.7
    ).add_to(m)

# 7) 路徑規劃
current = origin_adj
waypoints = [origin_adj]
full_draw_segments = []
steps = 0
status_log = []

print("\n開始路徑規劃...")
while True:
    if visible(current, dest_adj):
        waypoints.append(dest_adj)
        full_draw_segments.append((current, dest_adj))
        status_log.append("direct")
        print(f"  步驟 {steps}: 直達終點")
        break

    P = first_blocking_polygon(current, dest_adj, polys, COLLISION_PREP_M)
    if P is None:
        waypoints.append(dest_adj)
        full_draw_segments.append((current, dest_adj))
        status_log.append("direct-fallback")
        print(f"  步驟 {steps}: 無阻擋，直達")
        break

    print(f"  步驟 {steps}: 發現障礙物")
    
    # 視覺化障礙物
    P_buf_ring_wgs = to_wgs(to_metric(P).buffer(BUFFER_KM*1000.0))
    folium.GeoJson(
        convert_geom_to_pacific(P_buf_ring_wgs),
        name=f"障礙物 {steps}",
        style_function=lambda x: {"color":"#ff7f0e","weight":2,"fillOpacity":0.1}
    ).add_to(m)

    step_info = best_step_for_polygon(current, dest_adj, P, visible)
    status_log.append(step_info["status"])
    if step_info["status"] in ("arc","fallback"):
        wp = step_info["next_wp"]
        if wp is None: 
            print("    無可用路徑點，停止")
            break
        waypoints.append(wp)
        full_draw_segments.append((current, wp))
        current = wp
        print(f"    繞行至: ({wp[0]:.2f}°, {wp[1]:.2f}°)")
    else:
        print("    規劃失敗，停止")
        break

    steps += 1
    if MAX_STEPS is not None and steps >= MAX_STEPS:
        print(f"  達到最大步數 {MAX_STEPS}")
        break

# 最後檢查
if waypoints[-1] != dest_adj and visible(waypoints[-1], dest_adj):
    full_draw_segments.append((waypoints[-1], dest_adj))
    waypoints.append(dest_adj)

# 8) 繪製最終路徑（連續的粗藍線）
print(f"\n繪製 {len(full_draw_segments)} 段航線...")
for i, (a, b) in enumerate(full_draw_segments):
    draw_gc_polyline_continuous(
        m, a, b, step_km=DRAW_STEP_KM,
        color='#1f77b4', weight=5, opacity=0.9
    )
    print(f"  第 {i+1} 段: ({a[0]:.2f}°, {a[1]:.2f}°) → ({b[0]:.2f}°, {b[1]:.2f}°)")

# 航點標記
for i, wp in enumerate(waypoints[1:-1], 1):
    wp_pacific = [wp[1], normalize_lon_to_pacific_view(wp[0])]
    folium.CircleMarker(
        wp_pacific,
        radius=4,
        tooltip=f"航點 {i}<br>({wp[0]:.2f}°, {wp[1]:.2f}°)",
        color='blue',
        fill=True,
        fill_opacity=0.8
    ).add_to(m)

# 停止提示
if (MAX_STEPS is not None and steps>=MAX_STEPS) or waypoints[-1]!=dest_adj:
    last_pacific = [waypoints[-1][1], normalize_lon_to_pacific_view(waypoints[-1][0])]
    folium.Marker(
        last_pacific,
        tooltip="路徑未完成",
        icon=folium.Icon(color='orange', icon='exclamation-sign')
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

out_path = r"C:\Users\slab\Desktop\Slab Project\Stage2 ETA\route_pacific_view.html"
m.save(out_path)
print(f"\n 已儲存至: {out_path}")
print(f"總航程: {total_length_km(waypoints):.1f} 公里")
print(f"路徑狀態: {status_log}")

起點: (120.3181, 22.58425) 
終點: (103.5, 10.633333) 
太平洋視角: 起點=120.32°, 終點=103.50°
載入 579 個陸地多邊形


AttributeError: 'MultiPolygon' object has no attribute 'exterior'

### visualize

In [31]:
import webbrowser
webbrowser.open("route_pacific_LVS.html")

True